In [1]:
# !rm -rf logs/ # clear logs
# !rm -rf spice_out/

# Imports

In [2]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice again
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


In [ ]:
import sympy as sp
import logging

from pathlib import Path

from symxplorer.spice_engine            import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.optimization            import Nevergrad_Spice_Bode_Optimizer
from symxplorer.designer_tools.utils    import Frequency_Weight
from symxplorer.designer_tools.domains  import Project_Setup
from symxplorer.designer_tools.tf_models import Second_Order_BP_TF, cascade_tf

from symxplorer.logging import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

# Instantiations


In [ ]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

04:59:40 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
04:59:40 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-09-30_04-59-40.log
04:59:40 - SymXplorer: [INFO] 🔧 spicelib logger set to 50


In [ ]:
# s = sp.symbols("s")
# target_tf = (s + 1) / (s**2 + 24*s + 2)
fc=100e6
q=10
k_bp=1e3
filter_inst = Second_Order_BP_TF(q=q, fc=fc, k_bp=k_bp)
target_tf   = filter_inst.get_tf()
target_tf

20000000000.0*pi*s/(s**2 + 20000000.0*pi*s + 4.0e+16*pi**2)

In [ ]:
# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

04:59:41 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml
04:59:41 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: NGOpt, type=nevergrad, budget=100, random_seed=48
04:59:41 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
04:59:41 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
04:59:41 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
04:59:41 - SymXplorer.domains: [INFO] 	Number of target specs: 3
04:59:41 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=1e6, tolerance=10000.0, goal=GoalType.EXACT, sim_type=SimType.AC, enable=True)
04:59:41 - SymXplorer.domains: [INFO] 		- TargetSpec(name=q, target=10, tolerance=1, goal=GoalType.EXACT, sim_type=SimType.AC, enable=True)
04:59:41 - SymXplorer.domains: [INFO] 		- TargetSpec(name=gain, target=10, to

Project_Setup(name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(9.999999999999999e-06), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(9.999999999999999e-06), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(9.999999999999999e-06), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(9.999999999999999e-06), 'min_

In [ ]:
# (2) Create the Spice Simulator Wrapper
wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename= PROJECT_SETUP.ws_root / PROJECT_SETUP.netlist,
    output_folder=PROJECT_SETUP.ws_root / PROJECT_SETUP.outdir,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

04:59:41 - SymXplorer.spicelib: [WARNING] ⚠️ Output directory already exists, re-creating: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
04:59:41 - SymXplorer.spicelib: [INFO] --------------------------------------------------
04:59:41 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
04:59:41 - SymXplorer.spicelib: [INFO] 	📝 Project: Tunable-TIA
04:59:41 - SymXplorer.spicelib: [INFO] 	📜 Schematic: tb_ac
04:59:41 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
04:59:41 - SymXplorer.spicelib: [INFO] --------------------------------------------------
04:59:41 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
04:59:41 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
04:59:41 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['VSS', 'GND', 'VDD', 'Vbias', 'Von', 'Vop', 'In', 'Ip']
04:59:41 - SymXplorer.spicelib: [INFO] Te

In [ ]:
circuit_optimizer = Nevergrad_Spice_Bode_Optimizer(
    spicelib_wrapper=wrapper,
    target_tf=target_tf,
    output_node='vout',
    frequency_weight=Frequency_Weight(lower=fc/10, upper=fc*10),
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

# Sanity Check

In [ ]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

04:59:41 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
04:59:42 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...
04:59:42 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.log
04:59:42 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.raw
04:59:42 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
04:59:42 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

# Method Calls

In [ ]:
circuit_optimizer.parameterize()

Dict(x_dut_cap_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_w=Log{Cl(0,6,b),exp=2.15},x_dut_res_3_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_w': 10.000000000000002, 'x_dut_nfet_l': 50.0, 'x_dut_cap_w': 50.0, 'x_dut_cap_l': 50.0, 'x_dut_res_s_l': 50.0, 'x_dut_res_s_w': 50.0, 'x_dut_res_3_l': 50.0, 'x_dut_res_3_w': 50.0}

In [ ]:
circuit_optimizer.optimize()

04:59:42 - SymXplorer.optimizer: [INFO] Optimizer is set to NGOpt with budget = 100
Optimizing:   0%|          | 0/100 [00:00<?, ?trial/s]2025-09-30 04:59:42,487 - nevergrad.optimization.optimizerlib - NGOpt16 selected Cobyla optimizer.
2025-09-30 04:59:42,487 - nevergrad.optimization.optimizerlib - NGOpt selected Cobyla optimizer.
04:59:42 - SymXplorer.optimizer: [INFO] computing the target complex response for 20000000000.0*pi*s/(s**2 + 20000000.0*pi*s + 4.0e+16*pi**2)
Optimizing: 100%|██████████| 100/100 [00:35<00:00,  2.82trial/s]


[{'params': {'x_dut_nfet_w': 10.000000000000002,
   'x_dut_nfet_l': 50.0,
   'x_dut_cap_w': 50.0,
   'x_dut_cap_l': 50.0,
   'x_dut_res_s_l': 50.0,
   'x_dut_res_s_w': 50.0,
   'x_dut_res_3_l': 50.0,
   'x_dut_res_3_w': 50.0},
  'loss': np.float64(82713.47156280214)},
 {'params': {'x_dut_nfet_w': 10.000000000000002,
   'x_dut_nfet_l': 50.0,
   'x_dut_cap_w': 50.0,
   'x_dut_cap_l': 66.66666666666667,
   'x_dut_res_s_l': 50.0,
   'x_dut_res_s_w': 50.0,
   'x_dut_res_3_l': 50.0,
   'x_dut_res_3_w': 50.0},
  'loss': np.float64(82713.47156280349)},
 {'params': {'x_dut_nfet_w': 10.000000000000002,
   'x_dut_nfet_l': 50.0,
   'x_dut_cap_w': 66.66666666666667,
   'x_dut_cap_l': 50.0,
   'x_dut_res_s_l': 50.0,
   'x_dut_res_s_w': 50.0,
   'x_dut_res_3_l': 50.0,
   'x_dut_res_3_w': 50.0},
  'loss': np.float64(82713.47156280349)},
 {'params': {'x_dut_nfet_w': 10.000000000000002,
   'x_dut_nfet_l': 66.66666666666667,
   'x_dut_cap_w': 50.0,
   'x_dut_cap_l': 50.0,
   'x_dut_res_s_l': 50.0,
   'x_

In [ ]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

05:00:18 - SymXplorer.optimizer: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/loss_curve.html
05:00:18 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


In [ ]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss = out

05:00:18 - SymXplorer.optimizer: [INFO] best loss: 2440.3320742343535


In [ ]:
circuit_optimizer.plot_solution(best_param)

05:00:18 - SymXplorer.optimizer: [INFO] total loss: 2440.3320742343535
05:00:18 - SymXplorer.optimizer: [INFO] mag_loss 142.74257169877697, phase_loss 9165.90053114266
